In [10]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [11]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [12]:
lat = 35.01
long = -80.95

In [13]:
# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": lat,
	"longitude": long,
	"start_date": "2024-01-01",
	"end_date": "2026-02-03",
	"daily": ["temperature_2m_max", "temperature_2m_min", "sunrise", "sunset", "sunshine_duration", "daylight_duration"],
	"hourly": ["temperature_2m", "relative_humidity_2m", "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high", "dew_point_2m", "apparent_temperature", "precipitation"],
	"timezone": "GMT",
	"temperature_unit": "fahrenheit",
	"wind_speed_unit": "ms",
}
responses = openmeteo.weather_api(url, params=params)

In [14]:
# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

Coordinates: 35.0439338684082°N -80.95419311523438°E
Elevation: 195.0 m asl
Timezone: NoneNone
Timezone difference to GMT+0: 0s


In [15]:
# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(2).ValuesAsNumpy()
hourly_cloud_cover_low = hourly.Variables(3).ValuesAsNumpy()
hourly_cloud_cover_mid = hourly.Variables(4).ValuesAsNumpy()
hourly_cloud_cover_high = hourly.Variables(5).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(6).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(7).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(8).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["cloud_cover_low"] = hourly_cloud_cover_low
hourly_data["cloud_cover_mid"] = hourly_cloud_cover_mid
hourly_data["cloud_cover_high"] = hourly_cloud_cover_high
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation"] = hourly_precipitation

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n")
hourly_dataframe


Hourly data



,date,temperature_2m,relative_humidity_2m,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_high,dew_point_2m,apparent_temperature,precipitation
0,2024-01-01 00:00:00+00:00,43.970001,72.729584,11.0,0.0,0.0,11.0,35.779999,37.217846,0.0
1,2024-01-01 01:00:00+00:00,44.869999,66.608871,13.0,0.0,0.0,13.0,34.430000,37.747486,0.0
2,2024-01-01 02:00:00+00:00,44.419998,67.274971,42.0,0.0,0.0,42.0,34.250000,37.061298,0.0
3,2024-01-01 03:00:00+00:00,43.790001,68.172958,65.0,0.0,0.0,65.0,33.980000,36.339653,0.0
4,2024-01-01 04:00:00+00:00,43.970001,66.738991,74.0,0.0,0.0,74.0,33.619999,36.241253,0.0
...,...,...,...,...,...,...,...,...,...,...
18355,2026-02-03 19:00:00+00:00,42.980000,46.930569,100.0,0.0,97.0,100.0,24.080000,36.430222,0.0
18356,2026-02-03 20:00:00+00:00,43.070000,49.105846,100.0,0.0,97.0,100.0,25.250000,36.526768,0.0
18357,2026-02-03 21:00:00+00:00,42.620003,53.427078,100.0,0.0,46.0,100.0,26.869999,36.298119,0.0
18358,2026-02-03 22:00:00+00:00,41.360001,57.985313,100.0,0.0,20.0,100.0,27.680000,34.889828,0.0


In [16]:
# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
daily_sunrise = daily.Variables(2).ValuesInt64AsNumpy()
daily_sunset = daily.Variables(3).ValuesInt64AsNumpy()
daily_sunshine_duration = daily.Variables(4).ValuesAsNumpy()
daily_daylight_duration = daily.Variables(5).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["sunrise"] = daily_sunrise
daily_data["sunset"] = daily_sunset
daily_data["sunshine_duration"] = daily_sunshine_duration
daily_data["daylight_duration"] = daily_daylight_duration

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n") 
daily_dataframe


Daily data



,date,temperature_2m_max,temperature_2m_min,sunrise,sunset,sunshine_duration,daylight_duration
0,2024-01-01 00:00:00+00:00,52.880001,39.200001,1704112308,1704147754,26646.005859,35437.503906
1,2024-01-02 00:00:00+00:00,49.099998,30.559999,1704198719,1704234200,32114.134766,35472.726562
2,2024-01-03 00:00:00+00:00,49.009998,29.480000,1704285129,1704320649,20301.378906,35510.898438
3,2024-01-04 00:00:00+00:00,47.480000,33.529999,1704371536,1704407098,32203.144531,35551.949219
4,2024-01-05 00:00:00+00:00,45.139999,24.980000,1704457942,1704493548,32252.244141,35595.816406
...,...,...,...,...,...,...,...
760,2026-01-30 00:00:00+00:00,43.070000,24.080000,1769775829,1769813415,32400.000000,37561.714844
761,2026-01-31 00:00:00+00:00,34.160000,19.400000,1769862185,1769899876,0.000000,37667.304688
762,2026-02-01 00:00:00+00:00,29.389999,10.490002,1769948540,1769986338,34190.070312,37774.417969
763,2026-02-02 00:00:00+00:00,36.230000,5.900000,1770034893,1770072800,34393.515625,37882.917969
